<a href="https://colab.research.google.com/github/dgs2135/qs-rankings-analysis/blob/main/notebooks/01_standardize_schema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df17_22 = pd.read_csv('qs-world-university-rankings-2017-to-2022-V2.csv')
df23 = pd.read_csv('2023 QS World University Rankings.csv')
df24 = pd.read_csv('2024 QS World University Rankings 1.1 (For qs.com).csv')
df25 = pd.read_csv('qs-world-rankings-2025.csv')
df26 = pd.read_csv('2026 QS World University Rankings.csv')
df27 = pd.read_csv('2027 QS World University Rankings.csv')

In [ ]:
for name, df in {
    "17_22": df17_22,
    "23": df23,
    "24": df24,
    "25": df25,
    "26": df26,
    "27": df27
}.items():
    print("\n" + "="*50)
    print(name)
    print(df.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import re

def clean_numeric_rank(rank_val):
    """
    Converts rank strings to numeric midpoints.
    '601-650' -> 625.5
    '1401+' -> 1401
    '501' -> 501
    """
    if pd.isna(rank_val) or str(rank_val).strip() in ['-', '']:
        return np.nan

    rank_str = str(rank_val).replace(',', '').strip()

    # Handle ranges like 601-650
    range_match = re.findall(r'(\d+)', rank_str)
    if len(range_match) == 2:
        start, end = map(float, range_match)
        return (start + end) / 2
    elif len(range_match) == 1:
        return float(range_match[0])

    return np.nan

def standardize_countries(country):
    if pd.isna(country):
        return country
    c = str(country).strip()
    mapping = {
        'United States': 'USA',
        'United States of America': 'USA',
        'United Kingdom': 'UK',
        'Mainland China': 'China',
        'China (Mainland)': 'China'
    }
    return mapping.get(c, c)

# --- 1. Define Mappings ---
# Standard Schema: [university, year, country, Rank, overall_score, academic_reputation,
#                  employer_reputation, faculty_student, citations, sustainability]

# 2017-2022
d17_clean = df17_22.rename(columns={
    'university': 'university',
    'score': 'overall_score',
    'student_faculty_ratio': 'faculty_student',
    'rank_display': 'Rank'
}).copy()

# 2023
d23_clean = df23.assign(year=2023).rename(columns={
    'institution': 'university',
    'location': 'country',
    'score scaled': 'overall_score',
    'ar score': 'academic_reputation',
    'er score': 'employer_reputation',
    'fsr score': 'faculty_student',
    'cpf score': 'citations'
})

# 2024
d24_clean = df24.assign(year=2024).rename(columns={
    'Institution Name': 'university',
    'Country': 'country',
    '2024 RANK': 'Rank',
    'Overall SCORE': 'overall_score',
    'Academic Reputation Score': 'academic_reputation',
    'Employer Reputation Score': 'employer_reputation',
    'Faculty Student Score': 'faculty_student',
    'Citations per Faculty Score': 'citations',
    'Sustainability Score': 'sustainability'
})

# 2025
d25_clean = df25.assign(year=2025).rename(columns={
    'Institution Name': 'university',
    'Location Full': 'country',
    '2025 Rank': 'Rank',
    'QS Overall Score': 'overall_score',
    'Academic Reputation': 'academic_reputation',
    'Employer Reputation': 'employer_reputation',
    'Faculty Student': 'faculty_student',
    'Citations per Faculty': 'citations'
})

# 2026
d26_clean = df26.assign(year=2026).rename(columns={
    'Institution Name': 'university',
    'Country/Territory': 'country',
    '2026 Rank': 'Rank',
    'Overall SCORE': 'overall_score',
    'AR SCORE': 'academic_reputation',
    'ER SCORE': 'employer_reputation',
    'FSR SCORE': 'faculty_student',
    'CPF SCORE': 'citations',
    'SUS SCORE': 'sustainability'
})

# 2027
d27_clean = df27.assign(year=2027).rename(columns={
    'Name': 'university',
    'Country/Territory': 'country',
    'Rank': 'Rank',
    'Overall SCORE': 'overall_score',
    'AR SCORE': 'academic_reputation',
    'ER SCORE': 'employer_reputation',
    'FSR SCORE': 'faculty_student',
    'CPF SCORE': 'citations',
    'SUS SCORE': 'sustainability'
})

# --- 2. Merge ---
all_dfs = [d17_clean, d23_clean, d24_clean, d25_clean, d26_clean, d27_clean]
master_df = pd.concat(all_dfs, ignore_index=True, sort=False)

# --- 3. Cleaning & Standardizing ---
master_df['country'] = master_df['country'].apply(standardize_countries)
master_df['NumericRank'] = master_df['Rank'].apply(clean_numeric_rank)

# Select final columns
final_cols = [
    'university', 'year', 'country', 'Rank', 'NumericRank', 'overall_score',
    'academic_reputation', 'employer_reputation', 'faculty_student',
    'citations', 'sustainability'
]

# Filter to columns that exist and drop exact duplicates
master_df = master_df[[c for c in final_cols if c in master_df.columns]].drop_duplicates()

# --- 4. Reporting ---
print("MASTER DATASET REPORT")
print(f"Final Columns: {master_df.columns.tolist()}")
print("\nRecords per Year:")
print(master_df['year'].value_counts().sort_index())
print("\nMissing Values Summary:")
print(master_df.isnull().sum())

# --- 5. Export ---
master_df.to_csv('QS_Master_2017_2027.csv', index=False)
display(master_df.head())
print("\nSuccessfully exported to 'QS_Master_2017_2027.csv'")